# 01 — Build an insurance claims agent

Step-by-step walkthrough of a LangGraph **ReAct agent** for insurance claims support. Every piece is built directly in this notebook so the moving parts are explicit:

1. Load mock policy and FAQ data
2. Build a small FAQ semantic-search helper
3. Wrap three agent tools with `@tool`
4. Build the LLM (`COMPLEX_MODEL_NAME`, default `gpt-5`)
5. Assemble the agent with `create_react_agent`
6. Add multi-turn memory via a Redis checkpointer

The re-usable version of the same build lives in `demo/shared/insurance_bot.py`; `02_router_cache.ipynb` imports it from there.

**Prereqs**
- Redis reachable at `REDIS_URL` (default `redis://localhost:6379`)
- `.env` populated with `MODEL_API_KEY` and `COMPLEX_MODEL_NAME`
- `pip install -r ../scripts/requirements.txt`

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

REPO_ROOT = Path.cwd().parents[1]
load_dotenv(REPO_ROOT / ".env")

API_KEY = os.environ["MODEL_API_KEY"]
ENDPOINT = os.environ.get("MODEL_ENDPOINT", "https://api.openai.com")
COMPLEX_MODEL = os.environ.get("COMPLEX_MODEL_NAME", "gpt-5")
REDIS_URL = os.environ.get("REDIS_URL", "redis://localhost:6379")

DATA_DIR = Path.cwd() / "data"
print("model :", COMPLEX_MODEL)
print("redis :", REDIS_URL)
print("data  :", DATA_DIR)

## 1. Load mock policy and FAQ data

`data/policies.json` holds a handful of fake policies keyed by `policy_id`. `data/insurance_faq.json` holds short Q/A entries that stand in for a real knowledge base.

In [ ]:
import json

with open(DATA_DIR / "policies.json") as f:
    POLICIES = {p["policy_id"]: p for p in json.load(f)}

with open(DATA_DIR / "insurance_faq.json") as f:
    FAQ = json.load(f)

print(f"policies: {len(POLICIES)}  |  ids: {list(POLICIES)}")
print(f"faq rows: {len(FAQ)}  |  sample: {FAQ[0]['question']}")

## 2. FAQ semantic-search helper

For the demo we embed the FAQ questions once with `sentence-transformers` and keep the matrix in memory. In a production deployment this would become a Redis vector index — we keep it local here so the focus stays on the agent shape.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

EMBEDDER = SentenceTransformer("all-MiniLM-L6-v2")
FAQ_QUESTIONS = [row["question"] for row in FAQ]
FAQ_EMB = np.asarray(EMBEDDER.encode(FAQ_QUESTIONS, normalize_embeddings=True))

def search_faq_local(query: str, k: int = 3):
    q = EMBEDDER.encode([query], normalize_embeddings=True)
    scores = (FAQ_EMB @ q.T).ravel()
    top = np.argsort(-scores)[:k]
    return [
        {
            "question": FAQ[i]["question"],
            "answer": FAQ[i]["answer"],
            "category": FAQ[i]["category"],
            "score": float(scores[i]),
        }
        for i in top
    ]

for hit in search_faq_local("rental car while mine is in the shop"):
    print(f"[{hit['score']:.2f}] {hit['question']}")

## 3. Define the agent tools

Each tool is a plain Python function wrapped with `@tool`. LangGraph's ReAct loop decides when to call them based on the LLM's function-calling output — the docstring is the contract the model sees.

In [ ]:
from langchain_core.tools import tool

@tool
def search_faq(query: str) -> list[dict]:
    """Search insurance FAQ guidance. Returns top matches with answer text."""
    return search_faq_local(query, k=3)

@tool
def get_policy_details(policy_id: str) -> dict:
    """Look up a policy by id (e.g. 'AUTO-1001'). Returns coverages and deductibles."""
    if policy_id not in POLICIES:
        return {"error": f"policy {policy_id} not found", "known_ids": list(POLICIES)}
    return POLICIES[policy_id]

REQUIRED_DOCS = {
    "auto": [
        "date and location of incident",
        "policy number",
        "photos of damage",
        "incident description",
        "police or incident report number",
    ],
    "windshield": [
        "photos of the damaged glass",
        "policy number",
        "date of damage",
    ],
    "homeowners": [
        "photos of damage",
        "policy number",
        "description of what happened",
        "repair estimates",
        "receipts for temporary repairs",
    ],
}

@tool
def list_required_documents(claim_type: str) -> list[str]:
    """List typical documents needed for a claim type (e.g. 'auto', 'windshield', 'homeowners')."""
    return REQUIRED_DOCS.get(claim_type.lower(), [
        "policy number",
        "date and description of incident",
        "any photos or receipts related to the loss",
    ])

TOOLS = [search_faq, get_policy_details, list_required_documents]
print("tools:", [t.name for t in TOOLS])

## 4. Build the LLM

Wire `ChatOpenAI` against whatever `COMPLEX_MODEL_NAME` is set in `.env`. Swap `ENDPOINT` to point at OpenShift AI / vLLM / Azure / etc. when deploying — the rest of the notebook does not change.

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=COMPLEX_MODEL,
    api_key=API_KEY,
    base_url=f"{ENDPOINT.rstrip('/')}/v1",
)
llm

## 5. Assemble the ReAct agent

`create_react_agent` returns a compiled LangGraph graph with a tool-calling loop. The system prompt keeps the agent grounded in tool output and pushes it away from topics the proposal marks as out-of-scope (claim status, payouts, adjuster assignments).

In [ ]:
from langgraph.prebuilt import create_react_agent

AGENT_SYSTEM_PROMPT = (
    "You are an insurance claims assistant. "
    "Use the provided tools to look up FAQ guidance, policy details, and "
    "required documents before answering. "
    "Ground every answer in tool output. If a tool returns nothing relevant, "
    "say so plainly. Do not speculate about claim status, payouts, or "
    "adjuster assignments."
)

agent = create_react_agent(model=llm, tools=TOOLS, prompt=AGENT_SYSTEM_PROMPT)

from IPython.display import Image, display
display(Image(agent.get_graph().draw_mermaid_png()))

In [ ]:
question = "I rear-ended someone in my RAV4 this morning. What documents should I gather before I file the claim?"
result = agent.invoke({"messages": [{"role": "user", "content": question}]})
print(result["messages"][-1].content)

## 6. Multi-turn memory with Redis

LangGraph stores conversation state through a `checkpointer`. Plugging in `RedisSaver` from `langgraph-checkpoint-redis` means a given `thread_id` can be resumed across notebook runs — and, later, across pods on OpenShift.

`RedisSaver.from_conn_string` is a context manager, so we build the memory-backed agent inside the `with` block.

In [ ]:
from langgraph.checkpoint.redis import RedisSaver

thread_id = "jordan-auto-claim"

def ask(a, q, tid):
    out = a.invoke(
        {"messages": [{"role": "user", "content": q}]},
        config={"configurable": {"thread_id": tid}},
    )
    return out["messages"][-1].content

with RedisSaver.from_conn_string(REDIS_URL) as checkpointer:
    checkpointer.setup()
    mem_agent = create_react_agent(
        model=llm, tools=TOOLS, prompt=AGENT_SYSTEM_PROMPT, checkpointer=checkpointer
    )

    print("--- turn 1 ---")
    print(ask(mem_agent,
        "Hi, I'm the holder of policy AUTO-1001. I just had a minor fender bender. What do I need to do first?",
        thread_id))

    print("\n--- turn 2 ---")
    print(ask(mem_agent,
        "Got it. Do I need to pay my deductible up front, and how much is it on that policy?",
        thread_id))

    print("\n--- turn 3 ---")
    print(ask(mem_agent,
        "Also — while my car is being repaired, will I have a rental covered?",
        thread_id))

## Next

`02_router_cache.ipynb` wraps this agent with:

1. a **semantic router** so account/simple questions go to a cheap `gpt-4.1` call and prompt-injection attempts are refused
2. a **semantic cache** in front of the agent, populated only from 👍 user feedback

The re-usable version of the build above is `demo/shared/insurance_bot.py` — open it alongside this notebook to see the same pieces as a plain Python module.